# Test of voice command classification using deep learning

In [6]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import wave
import numpy as np
from torch.utils.data import DataLoader, Dataset

In [10]:
# ==========================================
# 1. CONFIGURATION & DATASET LOADING
# ==========================================
BATCH_SIZE = 64
SAMPLE_RATE = 16000
EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

class SubsetSC(Dataset):
    def __init__(self, subset: str = None):
        candidates = [
            Path("SpeechCommands/speech_commands_v0.02"),
            Path("notebooks/speech-command-classification/SpeechCommands/speech_commands_v0.02"),
        ]
        self.root = next((path for path in candidates if path.is_dir()), None)
        if self.root is None:
            raise FileNotFoundError("Speech Commands v0.02 dataset directory was not found")
        self._walker = sorted(
            str(path) for path in self.root.glob("*/*.wav")
            if not path.parent.name.startswith("_")
        )
        # Apply the official train/validation/test split lists.
        def load_list(filename):
            with (self.root / filename).open() as fileobj:
                return {str(self.root / line.strip()) for line in fileobj}

        if subset == "validation":
            self._walker = sorted(load_list("validation_list.txt"))
        elif subset == "testing":
            self._walker = sorted(load_list("testing_list.txt"))
        elif subset == "training":
            excludes = load_list("validation_list.txt") | load_list("testing_list.txt")
            self._walker = [w for w in self._walker if w not in excludes]

    def __len__(self):
        return len(self._walker)

    def get_metadata(self, index):
        path = self._walker[index]
        filename = Path(path).stem
        speaker_id, utterance_number = filename.rsplit("_nohash_", 1)
        return path, SAMPLE_RATE, Path(path).parent.name, speaker_id, int(utterance_number)

    def __getitem__(self, index):
        path, sample_rate, label, speaker_id, utterance_number = self.get_metadata(index)
        with wave.open(path, "rb") as audio_file:
            if audio_file.getsampwidth() != 2:
                raise ValueError(f"Expected 16-bit PCM audio in {path}")
            samples = np.frombuffer(audio_file.readframes(audio_file.getnframes()), dtype="<i2")
            waveform = samples.reshape(-1, audio_file.getnchannels()).T.copy()
        waveform = torch.from_numpy(waveform).to(torch.float32).div_(32768.0)
        return waveform, sample_rate, label, speaker_id, utterance_number

train_set = SubsetSC("training")
test_set = SubsetSC("testing")

# Target command labels (subset of 35 words or all depending on choice)
labels = sorted({Path(path).parent.name for path in train_set._walker})
word_to_idx = {word: i for i, word in enumerate(labels)}

def collate_fn(batch):
    tensors, targets = [], []
    for waveform, sample_rate, label, speaker_id, utterance_number in batch:
        # Ensure uniform length (1 second = 16,000 samples)
        if waveform.shape[1] < SAMPLE_RATE:
            pad = SAMPLE_RATE - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad))
        else:
            waveform = waveform[:, :SAMPLE_RATE]
        
        tensors.append(waveform)
        targets.append(word_to_idx[label])
        
    return torch.stack(tensors), torch.tensor(targets)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

Using device: cuda


In [11]:
# ==========================================
# 2. FEATURE EXTRACTION & MODEL ARCHITECTURE
# ==========================================
class MelSpectrogram(nn.Module):
    def __init__(self, sample_rate, n_mels=64, n_fft=400, hop_length=200):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.register_buffer("window", torch.hann_window(n_fft))
        frequencies = torch.linspace(0, sample_rate / 2, n_fft // 2 + 1)
        mel_max = 2595 * torch.log10(torch.tensor(1 + sample_rate / 2 / 700))
        mel_points = torch.linspace(0, mel_max, n_mels + 2)
        hz_points = 700 * (10 ** (mel_points / 2595) - 1)
        lower = (frequencies[None, :] - hz_points[:-2, None]) / (hz_points[1:-1] - hz_points[:-2])[:, None]
        upper = (hz_points[2:, None] - frequencies[None, :]) / (hz_points[2:] - hz_points[1:-1])[:, None]
        self.register_buffer("filterbank", torch.minimum(lower, upper).clamp(min=0))

    def forward(self, waveforms):
        spectrum = torch.stft(
            waveforms.squeeze(1), self.n_fft, self.hop_length,
            window=self.window, return_complex=True,
        ).abs().square()
        return torch.matmul(self.filterbank, spectrum).unsqueeze(1)

mel_transform = MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=64).to(DEVICE)

class AudioCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = AudioCNN(num_classes=len(labels)).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

ImportError: cannot import name 'skip_code' from 'torch._C._dynamo.eval_frame' (unknown location)

In [ ]:
# ==========================================
# 3. TRAINING LOOP
# ==========================================
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for waveforms, targets in train_loader:
        waveforms, targets = waveforms.to(DEVICE), targets.to(DEVICE)
        
        # Extract Mel-spectrograms on the fly
        spectrograms = mel_transform(waveforms)
        
        optimizer.zero_grad()
        outputs = model(spectrograms)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss / len(train_loader):.4f}")

In [ ]:
# # ==========================================
# # 4. LIVE MICROPHONE INFERENCE SANDBOX
# # ==========================================
# def listen_and_predict():
#     import sounddevice as sd
#     model.eval()
#     duration = 1.0  # 1-second sample window
#     print("\nListening for a command ('yes', 'no', 'stop', 'go', etc.)...")
    
#     audio_np = sd.rec(int(SAMPLE_RATE * duration), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
#     sd.wait()
    
#     waveform = torch.from_numpy(np.squeeze(audio_np)).unsqueeze(0).to(DEVICE)
#     spec = mel_transform(waveform).unsqueeze(0) # Add batch dim
    
#     with torch.no_grad():
#         output = model(spec)
#         pred_idx = torch.argmax(output, dim=1).item()
#         predicted_label = labels[pred_idx]
        
#     print(f"Detected Command: **{predicted_label.upper()}**")

# Uncomment to test live with your microphone:
# listen_and_predict()